In [3]:
import numpy as np
import pandas as pd
train = pd.read_csv('/content/sample_submission.csv')
test = pd.read_csv('/content/test.csv')
sample_submission = pd.read_csv('/content/train.csv')

In [4]:
train_dsn_mart = pd.read_csv('train.csv')
test_dsn_mart = pd.read_csv('test.csv')

In [5]:
train_dsn_mart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6818 entries, 0 to 6817
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   6818 non-null   object 
 1   product_code         6818 non-null   object 
 2   product_weight_kg    5593 non-null   float64
 3   fat_content          6818 non-null   object 
 4   shelf_visibility     6818 non-null   float64
 5   product_category     6818 non-null   object 
 6   product_price        6818 non-null   float64
 7   store_code           6818 non-null   object 
 8   store_age_years      6818 non-null   int64  
 9   store_size           4899 non-null   object 
 10  store_location_tier  6818 non-null   object 
 11  store_format         6818 non-null   object 
 12  total_sales          6818 non-null   float64
dtypes: float64(4), int64(1), object(8)
memory usage: 692.6+ KB


In [6]:
train_dsn_mart.head()

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format,total_sales
0,row_00000,PRD-PRFP9S,14.252,Low Fat,0.0271,Frozen Foods,81.37,STORE-AGY,45,Large,Tier_3,Standard Supermarket,1764.98
1,row_00001,PRD-PXXK71,7.698,Low Fat,0.0720,HEALTH AND HYGIENE,42.05,STORE-YLW,35,Small,Tier_1,Standard Supermarket,342.13
2,row_00002,PRD-V5MOIJ,14.264,Regular,0.0421,Canned,41.35,STORE-89Z,33,Medium,Tier_1,Standard Supermarket,378.85
3,row_00003,PRD-UN5Z3J,NaN,Regular,0.0449,soft drinks,174.35,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket,5595.72
4,row_00004,PRD-6RDQYB,10.338,Regular,0.0120,meat,203.06,STORE-9RG,28,Small,Tier_2,Standard Supermarket,2375.36


In [7]:
train_dsn_mart.describe()

,product_weight_kg,shelf_visibility,product_price,store_age_years,total_sales
count,5593.000000,6818.000000,6818.000000,6818.000000,6818.000000
mean,12.850876,0.065527,140.473623,34.178205,2174.756574
std,4.646542,0.051566,62.413513,8.384574,1697.894956
min,4.482000,0.000000,31.110000,23.000000,32.700000
25%,8.750000,0.026600,93.280000,28.000000,834.792500
50%,12.647000,0.053200,142.065000,33.000000,1790.890000
75%,16.894000,0.093400,185.987500,45.000000,3092.587500
max,21.883000,0.320100,273.040000,47.000000,12996.820000


In [8]:
from sklearn.model_selection import train_test_split
X = train_dsn_mart.drop(['id', 'product_code', 'product_category', 'total_sales'], axis=1)
y = train_dsn_mart['total_sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=21)

In [9]:
test_dsn_mart_1 = test_dsn_mart.drop(['id', 'product_code', 'product_category'], axis=1)

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import VotingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error


float_int_cols = ['product_weight_kg', 'shelf_visibility', 'product_price', 'store_age_years']

object_cols = ['fat_content', 'store_code', 'store_size', 'store_location_tier', 'store_format']

# imputation and encoding object to numbers
preprocessor = ColumnTransformer(transformers=[
    ('flint', Pipeline([('imputer', SimpleImputer(strategy='median'))]), float_int_cols),
    ('obj', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(drop='first', sparse_output=False))]), object_cols)
])

# Pipeline
comparison_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),

    # feature selection
    ('feature_selection', SelectFromModel(
        DecisionTreeRegressor(random_state=22),
        max_features=10,
        threshold=-float('inf')
    )),

    ('model', DecisionTreeRegressor(random_state=22))
])

# hyper-parameters search
param_grid = {
    'preprocessor__flint__imputer__strategy': ['mean', 'median'],
    'model__max_depth': [None, 5, 7],
    "model__min_samples_leaf": [10, 20]
}

# model fitting
grid_search = GridSearchCV(comparison_pipeline, param_grid, cv=10,
                           scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Save best model
top_model = grid_search.best_estimator_

# Model performance
y_pred = top_model.predict(X_test)
rmse_score = root_mean_squared_error(y_test, y_pred)

print(f"Root Mean Squared Error (RMSE): {rmse_score:.3f}")


Root Mean Squared Error (RMSE): 1127.894


In [11]:
solution = top_model.predict(test_dsn_mart_1)
solution_df = pd.DataFrame({'id': test['id'], 'total_sales': solution})
solution_df.to_csv('dtree.csv', index=False)


In [12]:
import xgboost as xgb

# Pipeline
comparison_pipeline_2 = Pipeline(steps=[
    ('preprocessor', preprocessor),

    # feature selection
    ('feature_selection', SelectFromModel(
        DecisionTreeRegressor(random_state=22),
        max_features=10,
        threshold=-float('inf')
    )),

('model', xgb.XGBRegressor(objective='reg:squarederror', random_state=22))
])

# hyper-parameters search
param_grid_2 = {
    'preprocessor__flint__imputer__strategy': ['mean', 'median'],
    'model__n_estimators': [50, 100, 150],
    'model__max_depth': [None, 5, 10]
}

# Model fitting
grid_search_2 = GridSearchCV(comparison_pipeline_2, param_grid=param_grid_2, cv=10,
                           scoring='neg_mean_squared_error', n_jobs=-1)
grid_search_2.fit(X_train, y_train)

# Save best model
top_model_2 = grid_search_2.best_estimator_

# Model performance
y_pred_2 = top_model_2.predict(X_test)
rmse_score_2 = root_mean_squared_error(y_test, y_pred_2)

print(f"Root Mean Squared Error (RMSE): {rmse_score_2:.3f}")


Root Mean Squared Error (RMSE): 1146.769


In [13]:
solution_2 = top_model_2.predict(test_dsn_mart_1)
solution_2_df = pd.DataFrame({'id': test['id'], 'total_sales': solution_2})
solution_2_df.to_csv('xgb.csv', index=False)

In [14]:
# Ensembling
top_model_3 = VotingRegressor(
    estimators=[
        ('dtree', top_model),
        ('xgb', top_model_2)
    ]
)

# Train and predict exactly like a single model
top_model_3.fit(X_train, y_train)
y_pred_3 = top_model_3.predict(X_test)

rmse_score_3 = root_mean_squared_error(y_test, y_pred_3)

print(f"Root Mean Squared Error (RMSE): {rmse_score_3:.3f}")

Root Mean Squared Error (RMSE): 1123.471


In [15]:
solution_3 = top_model_3.predict(test_dsn_mart_1)
solution_3_df = pd.DataFrame({'id': test['id'], 'total_sales': solution_3})
solution_3_df.to_csv('dtree_xgb.csv', index=False)